# Streaming in LangGraph

Streaming makes agent systems feel alive. Instead of waiting for the whole graph to finish, we can:

- show partial results in real time
- surface progress while tools and models run
- debug which node produced which output

LangGraph exposes **two streaming APIs**:

1. **Stream-mode API**: `graph.stream(...)` — practical for notebook demos and app UIs
2. **Event Streaming API**: `graph.stream_events(...)` — lower-level event feed for node lifecycle events, tracing, and advanced observability

In this notebook, we will focus on `graph.stream(...)` and compare the major `stream_mode` options.


In [ ]:
%run ../../langchain_common.py

## Build a Demo Agent

We will create a small ReAct-style arithmetic agent with two tools:

- `add(a, b)`
- `multiply(a, b)`

The graph uses:

- `llm_noreason` for tool-calling
- `ToolNode` to execute tool calls
- `tools_condition` to route either to tools or to the end of the run

This gives us a compact graph that is perfect for streaming demos.


In [ ]:
from pprint import pprint

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image, display


In [ ]:
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


tools = [add, multiply]
llm_with_tools = llm_noreason.bind_tools(tools, parallel_tool_calls=False)
system_message = SystemMessage(
    content=(
        "You are a helpful arithmetic assistant. "
        "Use tools for calculations and then answer clearly."
    )
)


def assistant(state: MessagesState):
    response = llm_with_tools.invoke([system_message] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

graph = builder.compile()
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))


## `stream_mode="values"`

`values` yields the **full graph state after each step**.

This is the easiest mode to understand because every chunk is a complete snapshot. It is useful when:

- you want the whole state each time
- your state is small
- you are learning how the graph evolves


In [ ]:
inputs = {
    "messages": [
        HumanMessage(content="Add 2 and 3, then multiply the result by 4.")
    ]
}

for i, chunk in enumerate(graph.stream(inputs, stream_mode="values"), start=1):
    print(f"--- values chunk {i} ---")
    print("state keys:", list(chunk.keys()))
    print("message count:", len(chunk["messages"]))
    print("last message type:", type(chunk["messages"][-1]).__name__)
    chunk["messages"][-1].pretty_print()


Notice that each `values` chunk contains the *entire* `messages` list up to that point. That makes inspection easy, but it can be more expensive than streaming only the changes.

## `stream_mode="updates"`

`updates` yields only the **delta** produced by each node.

This is usually the most practical mode because it avoids resending the full state every time. It is a great default for dashboards, logs, and simple streaming UIs.


In [ ]:
for i, chunk in enumerate(graph.stream(inputs, stream_mode="updates"), start=1):
    print(f"--- updates chunk {i} ---")
    pprint(chunk)


Compared with `values`, the `updates` chunks are smaller. Instead of a full snapshot, you only receive what each node added or changed.

## `stream_mode="messages"`

`messages` streams LLM output at the token/message-chunk level. This is especially useful for chat apps, where users should see the response appear incrementally.

Pattern to remember:

```python
for msg, metadata in graph.stream(inputs, stream_mode="messages"):
    ...
```


In [ ]:
for msg, metadata in graph.stream(inputs, stream_mode="messages"):
    if msg.content:
        text = msg.content if isinstance(msg.content, str) else str(msg.content)
        print(text, end="")
print()


In [ ]:
for msg, metadata in graph.stream(inputs, stream_mode="messages"):
    if msg.content:
        print("\nTOKEN/CHUNK:", repr(msg.content))
        print("metadata keys:", list(metadata.keys()))
        break


## Multiple Stream Modes

You can request more than one mode at once by passing a list. LangGraph then labels each emitted chunk with its stream type, so you can route different chunk kinds to different parts of your application.


In [ ]:
for mode, chunk in graph.stream(inputs, stream_mode=["updates", "messages"]):
    if mode == "updates":
        print("\n[UPDATE]")
        pprint(chunk)
    elif mode == "messages" and chunk[0].content:
        text = chunk[0].content if isinstance(chunk[0].content, str) else str(chunk[0].content)
        print(text, end="")
print()


A common pattern is:

- send `messages` chunks to the chat window
- send `updates` chunks to a debug panel or activity log


## Custom Streaming

Sometimes you want to emit your own progress updates from inside a node. LangGraph supports this with `get_stream_writer()`.

This is perfect for status messages such as:

- `"Loading documents..."`
- `"Calling retriever..."`
- `"Scoring candidates..."`


In [ ]:
from langgraph.config import get_stream_writer
from langgraph.graph import END


def progress_node(state: MessagesState):
    writer = get_stream_writer()
    writer("Processing step 1...")

    user_request = state["messages"][-1].content

    writer("Processing step 2...")
    response = llm_noreason.invoke(
        [
            SystemMessage(content="Answer briefly and clearly."),
            HumanMessage(content=f"Summarize this request in one sentence: {user_request}"),
        ]
    )

    writer("Processing step 3...")
    return {"messages": [response]}


custom_builder = StateGraph(MessagesState)
custom_builder.add_node("progress_node", progress_node)
custom_builder.add_edge(START, "progress_node")
custom_builder.add_edge("progress_node", END)
custom_graph = custom_builder.compile()


In [ ]:
custom_inputs = {
    "messages": [HumanMessage(content="Explain why streaming improves user experience.")]
}

for chunk in custom_graph.stream(custom_inputs, stream_mode="custom"):
    print(chunk)


## Key Takeaways

| Stream mode / API | What you get | Best use case |
|---|---|---|
| `values` | Full state snapshot after each step | Learning, debugging, inspecting small states |
| `updates` | Only state deltas | Default choice for most applications |
| `messages` | Incremental LLM message/token chunks | Chat UIs and live text generation |
| `custom` | Your own node-emitted progress updates | Status bars, pipeline progress, custom telemetry |
| `graph.stream_events(...)` | Low-level runtime events | Advanced debugging, tracing, observability |

If you are unsure where to start, use **`updates`**. Add **`messages`** when you want token-by-token UI streaming, and add **`custom`** when your nodes need to report their own progress.
